In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API Key is set") 

API Key is set


In [2]:
from langchain_openai import ChatOpenAI


In [3]:
llm = ChatOpenAI(model="gpt-5-nano", temperature = 0)

In [4]:
response =llm.invoke("What is AI? Tell me in 1 line")

response.content

'Artificial intelligence is the field of computer science that builds systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, and problem-solving.'

## **RAG IMPLEMENTATION WITH PDF**


#### **Extracting Text from Pdf**

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ("./Docs/fabric-admin.pdf")

loader = PyPDFLoader(pdf_path)

docs = loader.load()

docs


[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26056.01', 'creator': 'Microsoft Learn', 'creationdate': '2026-03-28T14:53:03+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2026-03-28T14:53:03+00:00', 'source': './Docs/fabric-admin.pdf', 'total_pages': 308, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nFabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1

#### Creating own Metadata for pdf chunks

In [7]:
for i in docs:
    i.metadata = {"source": "fabric-admin.pdf",
                  "developer": "Microsoft"}

In [8]:
docs[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}

#### **STEP 2: SPLITTING THE DOCUMENT INTO CHUNKS**

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Tell us about your PDF experience.\nFabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usa

In [10]:
len(chunks)

347

### **STEP 3: CREATING EMBEDDINGS FOR THE CHUNKS**

In [11]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [12]:
#embedding_model.embed_query("WHAT IS AI?")

### **STEP 4" CREATE AND STORE EMBEDDINGS IN EXISTING Local VECTOR STORE**

In [12]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    persist_directory= "./Vector_Store",
    embedding_function= embedding_model
    )

/var/folders/8k/qsf58g2j0fnf_69x24jykdth0000gn/T/ipykernel_47355/454389501.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [13]:
vector_store.add_documents(chunks)

['9f799272-70c6-413c-93cf-b7309ee071e3',
 '89c8b965-9c48-486d-989b-5019fa88fac8',
 'b6aaefe4-4cf3-4f4e-b297-8130dac185c5',
 '838ab63d-c2f3-4f56-a8af-279c19d63a10',
 'a89c2781-82ce-4629-9dd4-39f0112926e3',
 '17b06d40-5219-4200-83a0-8135b8dce2a7',
 '1fe31e78-de04-4743-9801-fb2951b7f8fd',
 '7eafbc44-209c-443d-b79c-7b0cbcc98d30',
 'a8551747-f117-40ca-8446-bc32adb69675',
 '9bcd7123-1cf4-4046-868b-53da0e04669d',
 'a292b377-58f8-46cc-ae7d-ae32958fa39e',
 '2f313405-82ad-44d6-8908-0114d8d237a6',
 'fed925e6-80ae-40a5-81af-8d06a45eb0a4',
 '44b3c9d9-ec46-4666-b3fc-926e3dc9d3c7',
 'c88289f8-9808-421e-a0d8-44344510126a',
 '65928eff-5825-4eef-8fc3-322af3db0b75',
 '5a317698-509b-44bd-bb25-e03b8f1b544a',
 '9f941c7e-374b-4189-92d5-d586e3afeb9c',
 '9afb5250-9b90-4b87-8766-52f372dec885',
 'c13c9993-cc07-408c-8dca-c898e7c25c23',
 'f66f708d-af3b-40ed-984a-52be38dd94db',
 '37f873f1-0190-46b1-bca6-acbcec65edeb',
 '7e33975a-6d13-40b2-b0d7-05be0e7bb84b',
 'f342eca3-46ee-46b0-bd51-f51a11e1402d',
 '26f3516c-0266-

In [13]:
#vectors = []
#for doc in chunks:
#    emb = embedding_model.embed_documents(doc.page_content)
#    vectors.append(emb)

#vectors

#### **STEP 5: SEMANTIC SEARCH**

In [14]:
vector_store.similarity_search("What is Microsoft Fabric Admin?", k = 3)

[Document(metadata={'developer': 'Microsoft', 'source': 'fabric-admin.pdf'}, page_content='What is Microsoft Fabric admin?\n07/01/2024\nMicrosoft Fabric admin is the management of the organization-wide settings that control how\nMicrosoft Fabric works. Users that are assigned to admin roles configure, monitor, and\nprovision organizational resources. This article provides an overview of admin roles, tasks, and\ntools to help you get started.\nThere are several roles that work together to administer Microsoft Fabric for your organization.\nMost admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. The\ncapacity admin roles are assigned when the capacity is created. To learn more about each of\nthe admin roles, see About admin roles. To learn how to assign admin roles, see Assign admin\nroles.\nThis section lists the Microsoft 365 admin roles and the tasks they can perform.\nGlobal administrator\nUnlimited access to all management features for the organizatio

#### **Reuse Vector db**

In [7]:
vectorstore_persist = Chroma(
    persist_directory= "./Vector_Store",
    embedding_function= embedding_model
)

/var/folders/8k/qsf58g2j0fnf_69x24jykdth0000gn/T/ipykernel_46838/1683479691.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_persist = Chroma(


In [8]:
vectorstore_persist.similarity_search("What do we need onelake?", k = 3)

[Document(metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft'}, page_content="OneLake uses zone-redundant storage (ZRS) where available to protect against datacenter\nfailures. You can also enable business continuity and disaster recovery (BCDR) for a capacity to\nreplicate your data to a secondary geographic region.\nFor more information, see Plan for disaster recovery and data protection.\nReady to start using OneLake? Here's how to get started:\nOneLake file explorer for Windows\nOneLake shortcuts\nRecover deleted files in OneLake\nOneLake access and APIs\nLast updated on 02/12/2026\nNext steps\nCreate your first lakehouse with OneLake\nRelated content"),
 Document(metadata={'developer': 'Microsoft', 'source': 'fabric-onelake.pdf'}, page_content="Since OneLake supports the same APIs as ADLS and Blob Storage, many open source\nlibraries and packages compatible with ADLS and Blob Storage work seamlessly with\nOneLake (for example, Azure Storage Explorer). Other librarie